In [ ]:
from geo_data import data_handler, helpers, svg_handler
import re
import subprocess
import pandas as pd

In [ ]:
countries = data_handler.load("country", "ne", resolution=110)

regional_groups = data_handler.get_regional_groups()
country_transl = data_handler.get_country_translations()

In [ ]:
def get_country_codes(country_names):
    country_codes = []
    for c in country_names:
        country_code = country_transl[country_transl["german"] == c]["code"].iloc[0]
        country_codes.append(country_code)
    assert len(country_names) == len(country_codes)
    return country_codes


def get_group_id(group_name):
    group_id = group_name.lower()
    umlaut_map = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"})
    group_id = group_id.translate(umlaut_map)
    group_id = re.sub(r"[^a-zA-Z0-9._-]+", "_", group_id)
    return group_id


save_path_orig = (
    helpers.get_top_directory() / "results" / "regional_groups" / "original"
)
save_path_optim = (
    helpers.get_top_directory() / "results" / "regional_groups" / "optimized"
)

# get countries belonging to a regional group
for reg_group_name, reg_group in regional_groups.items():
    reg_group_id = get_group_id(reg_group_name)

    # get regional group core countries
    reg_country_codes = get_country_codes(reg_group["core"])
    reg_mask = countries["adm0_a3_de"].isin(reg_country_codes)
    reg_countries = countries[reg_mask]
    # get regional group optional countries
    opt_country_codes = get_country_codes(reg_group["optional"])
    opt_mask = countries["adm0_a3_de"].isin(opt_country_codes)
    opt_countries = countries[opt_mask]
    # get all other countries
    other_countries = countries[~(reg_mask | opt_mask)]

    # determine center of regional group
    center = reg_countries.union_all().centroid

    if reg_group["projection"] == "ortho":
        # draw SVG canvas
        canvas = svg_handler.OrthoMapSVG(width=500, center=(center.x, center.y))
        canvas.add_sea()
        # other countries
        land_kwargs = canvas.get_kwargs("land")
        canvas.add_gdf(other_countries, "countries", **land_kwargs)
        # highlighted countries
        highlight_color = svg_handler.COLORS["highlight"]
        land_color = land_kwargs["fill"]
        land_kwargs["fill"] = highlight_color
        canvas.add_gdf(reg_countries, reg_group_id, **land_kwargs)
        # optional striped countries
        canvas.add_def(
            svg_handler.DiagonalStripedPattern(
                id="optional-country-pattern", color=(land_color, highlight_color)
            )
        )
        land_kwargs["fill"] = "url(#optional-country-pattern)"
        canvas.add_gdf(opt_countries, f"{reg_group_id}_optional", **land_kwargs)
        canvas.add_shadow()
    else:
        # draw SVG canvas
        canvas = svg_handler.MapSVG(width=500)
        canvas.add_background(svg_handler.COLORS["lake"])
        # other countries
        land_kwargs = canvas.get_kwargs("land")
        canvas.add_gdf(other_countries, "countries", **land_kwargs)
        # highlighted countries
        highlight_color = svg_handler.COLORS["highlight"]
        land_color = land_kwargs["fill"]
        land_kwargs["fill"] = highlight_color
        canvas.add_gdf(reg_countries, reg_group_id, **land_kwargs)
        # optional striped countries
        canvas.add_def(
            svg_handler.DiagonalStripedPattern(
                id="optional-country-pattern", color=(land_color, highlight_color)
            )
        )
        land_kwargs["fill"] = "url(#optional-country-pattern)"
        canvas.add_gdf(opt_countries, f"{reg_group_id}_optional", **land_kwargs)

    # save svg
    canvas.save(save_path_orig / f"{reg_group_id}.svg")
    # optimize saved svg
    subprocess.run(
        [
            "svgo",
            str(save_path_orig / f"{reg_group_id}.svg"),
            "-o",
            str(save_path_optim / f"{reg_group_id}.svg"),
        ]
    )

In [ ]:
# create a csv file that can be imported by anki
reg_group_dicts = []
for reg_group_name, reg_group in regional_groups.items():
    reg_group_id = get_group_id(reg_group_name)

    core, optional = reg_group["core"], reg_group["optional"]
    core_str, optional_str = ", ".join(core), ", ".join(optional)
    if optional_str != "":
        country_str = f"{core_str}, ({optional_str})"
    else:
        country_str = core_str

    file_name = f'<img src="{reg_group_id}.svg">'

    reg_group_dicts.append(
        {"Name": reg_group_name, "Countries": country_str, "Map": file_name}
    )
df = pd.DataFrame(reg_group_dicts)

save_path_csv = (
    helpers.get_top_directory() / "results" / "regional_groups" / "regional_groups.csv"
)
df.to_csv(save_path_csv, index=False, header=False)
df